# 💬 WhatsApp Chat Analyzer — Advanced Jupyter Notebook
> Full exploratory analysis of WhatsApp chat exports.  
> Includes: preprocessing, timelines, activity maps, NLP, emoji & link analysis, sentiment, and more.

---


## 1. 📦 Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy matplotlib seaborn plotly wordcloud emoji urlextract Pillow

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud, STOPWORDS
from collections import Counter
import emoji
from urlextract import URLExtract
import warnings
warnings.filterwarnings('ignore')

# Styling
plt.style.use('dark_background')
sns.set_theme(style='darkgrid')
PALETTE = ['#25D366','#128C7E','#075E54','#34B7F1','#00BCD4','#4CAF50','#FFC107','#FF9800']
extract = URLExtract()

print("✅ All libraries imported successfully!")


## 2. 📂 Load & Preprocess Chat File

In [ ]:
# ─── CHANGE THIS PATH ───
CHAT_FILE = 'Familygroup.txt'   # or WhatsApp Chat with XYZ.txt

with open(CHAT_FILE, 'r', encoding='utf-8') as f:
    data = f.read()

print(f"📄 File loaded — {len(data):,} characters")


In [ ]:
def preprocess(data):
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s-\s',
        r'\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s[APap][Mm]\s-\s',
        r'\[\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}:\d{2}\s[APap][Mm]\]\s',
        r'\d{1,2}-\d{1,2}-\d{2,4},\s\d{1,2}:\d{2}\s-\s',
    ]
    matched = next((p for p in patterns if re.findall(p, data)), patterns[0])
    messages = re.split(matched, data)[1:]
    dates = [d.strip().strip('[]').strip(' -').strip() for d in re.findall(matched, data)]

    df = pd.DataFrame({'user_message': messages, 'message_date': dates})

    date_formats = [
        '%d/%m/%Y, %H:%M', '%d/%m/%y, %H:%M',
        '%m/%d/%Y, %H:%M', '%m/%d/%y, %H:%M',
        '%d/%m/%Y, %I:%M %p', '%d/%m/%y, %I:%M %p',
    ]
    for fmt in date_formats:
        try:
            df['message_date'] = pd.to_datetime(df['message_date'], format=fmt, errors='raise')
            break
        except Exception:
            continue
    else:
        df['message_date'] = pd.to_datetime(df['message_date'], infer_datetime_format=True, errors='coerce')

    df.rename(columns={'message_date': 'date'}, inplace=True)

    users, msgs = [], []
    for msg in df['user_message']:
        entry = re.split(r'([\w\W]+?):\s', msg)
        if entry[1:]:
            users.append(entry[1])
            msgs.append(' '.join(entry[2:]))
        else:
            users.append('group_notification')
            msgs.append(entry[0])

    df['user'] = users
    df['message'] = msgs
    df.drop(columns=['user_message'], inplace=True)
    df.dropna(subset=['date'], inplace=True)

    df['only_date']  = df['date'].dt.date
    df['year']       = df['date'].dt.year
    df['month_num']  = df['date'].dt.month
    df['month']      = df['date'].dt.month_name()
    df['day']        = df['date'].dt.day
    df['day_name']   = df['date'].dt.day_name()
    df['hour']       = df['date'].dt.hour
    df['minute']     = df['date'].dt.minute
    df['quarter']    = df['date'].dt.quarter
    df['msg_length'] = df['message'].apply(len)
    df['word_count'] = df['message'].apply(lambda x: len(x.split()))
    df['is_media']   = df['message'].str.contains('<Media omitted>', na=False)
    df['is_deleted'] = df['message'].str.contains('This message was deleted|message was deleted', na=False)

    period = []
    for h in df['hour']:
        period.append('23-00' if h == 23 else '00-01' if h == 0 else f'{str(h).zfill(2)}-{str(h+1).zfill(2)}')
    df['period'] = period

    return df

df = preprocess(data)
print(f"✅ DataFrame shape: {df.shape}")
df.head(10)


## 3. 📊 Dataset Overview

In [ ]:
print("=== Dataset Info ===")
print(f"Total messages : {len(df):,}")
print(f"Date range     : {df['only_date'].min()} → {df['only_date'].max()}")
print(f"Participants   : {df[df['user'] != 'group_notification']['user'].nunique()}")
print(f"Media messages : {df['is_media'].sum():,}")
print(f"Deleted msgs   : {df['is_deleted'].sum():,}")
print()
print("=== Column Types ===")
print(df.dtypes)


In [ ]:
df.describe(include='all').T


## 4. 📈 Timeline Analysis

In [ ]:
# Daily Timeline
daily = df.groupby('only_date').count()['message'].reset_index()
daily.columns = ['date', 'count']
daily['rolling_7'] = daily['count'].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(16, 4))
ax.fill_between(daily['date'], daily['count'], alpha=0.3, color='#25D366')
ax.plot(daily['date'], daily['count'], color='#25D366', linewidth=1, alpha=0.7, label='Daily')
ax.plot(daily['date'], daily['rolling_7'], color='#FFC107', linewidth=2.5, label='7-Day Avg')
ax.set_title('Daily Messages + 7-Day Rolling Average', fontsize=14, pad=12)
ax.set_xlabel('Date'); ax.set_ylabel('Messages')
ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Monthly Timeline
monthly = df.groupby(['year','month_num','month']).count()['message'].reset_index()
monthly['time'] = monthly['month'].str[:3] + ' ' + monthly['year'].astype(str)

fig, ax = plt.subplots(figsize=(16, 4))
bars = ax.bar(monthly['time'], monthly['message'], color='#25D366', edgecolor='#075E54', linewidth=0.5)
ax.set_title('Monthly Message Volume', fontsize=14, pad=12)
plt.xticks(rotation=45, ha='right')
ax.set_ylabel('Messages')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=7)
plt.tight_layout(); plt.show()


## 5. 🗓️ Activity Maps

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

busy_day   = df['day_name'].value_counts().reindex(day_order, fill_value=0)
busy_month = df['month'].value_counts().reindex(month_order, fill_value=0)
hourly     = df.groupby('hour').count()['message']

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].bar(busy_day.index, busy_day.values, color=PALETTE[:7])
axes[0].set_title('Busiest Day of Week'); axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(busy_month.index, busy_month.values, color=PALETTE[:12])
axes[1].set_title('Busiest Month'); axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(hourly.index, hourly.values, color='#34B7F1')
axes[2].set_title('Hourly Activity'); axes[2].set_xlabel('Hour of Day')

plt.tight_layout(); plt.show()


In [ ]:
# Heatmap
heatmap_df = df.pivot_table(index='day_name', columns='period', values='message', aggfunc='count').fillna(0)
heatmap_df = heatmap_df.reindex([d for d in day_order if d in heatmap_df.index])

plt.figure(figsize=(18, 5))
sns.heatmap(heatmap_df, cmap='Greens', linewidths=0.3, linecolor='#222',
            cbar_kws={'label': 'Messages'})
plt.title('Weekly Activity Heatmap (Day × Hour Period)', fontsize=14, pad=12)
plt.tight_layout(); plt.show()


## 6. 👥 User Analysis

In [ ]:
users_df = df[df['user'] != 'group_notification']
top_users = users_df['user'].value_counts().head(10)
pct_df = round(top_users / len(users_df) * 100, 2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top_users.index[::-1], top_users.values[::-1], color=PALETTE[:10])
axes[0].set_title('Top 10 Most Active Users')
axes[0].set_xlabel('Messages')

axes[1].pie(top_users.values, labels=top_users.index, autopct='%1.1f%%',
            colors=PALETTE[:10], startangle=90)
axes[1].set_title('Message Share (%)')

plt.tight_layout(); plt.show()
print(pct_df)


In [ ]:
# Avg message length per user
avg_len = users_df.groupby('user')['msg_length'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 4))
plt.barh(avg_len.index[::-1], avg_len.values[::-1], color='#34B7F1')
plt.title('Average Message Length by User (characters)')
plt.xlabel('Avg Characters')
plt.tight_layout(); plt.show()


## 7. ☁️ Word Cloud & Text Analysis

In [ ]:
STOP_WORDS = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','it','this','that','was','are','be','have','he','she','they','we',
    'you','i','my','your','our','their','its','will','do','did','not','so',
    'Media','omitted','null','message','deleted','ok','yes','yeah','hi',
    'hello','hey','lol','haha','like','know','good','well','right','sure',
])

clean_msgs = df[~df['is_media'] & ~df['is_deleted']]['message'].str.cat(sep=' ')
words = [w for w in clean_msgs.lower().split() if w not in STOP_WORDS and len(w) > 2]
clean_text = ' '.join(words)

wc = WordCloud(width=1200, height=500, background_color='#111',
               colormap='Greens', max_words=200, collocations=False)
wc_img = wc.generate(clean_text)

plt.figure(figsize=(16, 6))
plt.imshow(wc_img, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Most Frequent Words', fontsize=14, pad=12)
plt.tight_layout(); plt.show()


In [ ]:
# Top words bar chart
word_freq = Counter(words).most_common(20)
wf_df = pd.DataFrame(word_freq, columns=['word','count'])

plt.figure(figsize=(12, 5))
plt.barh(wf_df['word'][::-1], wf_df['count'][::-1], color='#25D366')
plt.title('Top 20 Most Common Words')
plt.xlabel('Frequency')
plt.tight_layout(); plt.show()

wf_df


In [ ]:
# Bigrams
bigrams = []
for msg in df[~df['is_media'] & ~df['is_deleted']]['message']:
    ws = [w for w in msg.lower().split() if w not in STOP_WORDS and len(w) > 2 and w.isalpha()]
    bigrams.extend([f'{ws[i]} {ws[i+1]}' for i in range(len(ws)-1)])

bg_df = pd.DataFrame(Counter(bigrams).most_common(15), columns=['bigram','count'])

plt.figure(figsize=(12, 5))
plt.barh(bg_df['bigram'][::-1], bg_df['count'][::-1], color='#34B7F1')
plt.title('Top 15 Bigrams (Word Pairs)')
plt.xlabel('Frequency')
plt.tight_layout(); plt.show()

bg_df


## 8. 😀 Emoji Analysis

In [ ]:
all_emojis = []
for msg in df['message']:
    all_emojis.extend([c for c in msg if c in emoji.EMOJI_DATA])

emoji_df = pd.DataFrame(Counter(all_emojis).most_common(20), columns=['emoji','count'])
print(f"Total emojis used: {len(all_emojis):,}")
print(f"Unique emojis    : {len(set(all_emojis)):,}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(emoji_df['emoji'][::-1], emoji_df['count'][::-1], color=PALETTE[:20])
axes[0].set_title('Top 20 Emojis')

axes[1].pie(emoji_df['count'].head(8), labels=emoji_df['emoji'].head(8),
            autopct='%1.1f%%', colors=PALETTE[:8], startangle=90)
axes[1].set_title('Emoji Share (Top 8)')

plt.tight_layout(); plt.show()
emoji_df


In [ ]:
# Emoji per user
emoji_by_user = {}
for user in df[df['user'] != 'group_notification']['user'].unique():
    umsgs = df[df['user'] == user]['message']
    cnt = sum(1 for msg in umsgs for c in msg if c in emoji.EMOJI_DATA)
    emoji_by_user[user] = cnt

eu = pd.Series(emoji_by_user).sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 4))
plt.barh(eu.index[::-1], eu.values[::-1], color='#FFC107')
plt.title('Emoji Usage by User')
plt.xlabel('Total Emojis')
plt.tight_layout(); plt.show()


## 9. 🔗 Link Analysis

In [ ]:
domains = []
for msg in df['message']:
    for url in extract.find_urls(msg):
        try:
            domain = re.sub(r'^www\.', '', url.split('/')[2])
            domains.append(domain)
        except Exception:
            pass

domain_df = pd.DataFrame(Counter(domains).most_common(10), columns=['domain','count'])
print(f"Total links: {len(domains)}")

if not domain_df.empty:
    plt.figure(figsize=(10, 4))
    plt.barh(domain_df['domain'][::-1], domain_df['count'][::-1], color='#34B7F1')
    plt.title('Top Shared Domains')
    plt.xlabel('Times Shared')
    plt.tight_layout(); plt.show()

domain_df


## 10. 😊 Basic Sentiment Analysis

In [ ]:
positive_words = set(['good','great','love','awesome','amazing','wonderful','happy',
    'excellent','fantastic','nice','best','thanks','thank','beautiful','perfect',
    'brilliant','superb','congratulations','congrats','well','yay','yes'])
negative_words = set(['bad','sad','hate','terrible','awful','horrible','worst',
    'poor','wrong','failed','fail','sorry','angry','upset',
    'unfortunately','problem','issue','error','broken','never'])

pos, neg, neu = 0, 0, 0
for msg in df['message']:
    ws = set(msg.lower().split())
    p = len(ws & positive_words); n = len(ws & negative_words)
    if p > n: pos += 1
    elif n > p: neg += 1
    else: neu += 1

total = pos + neg + neu
sentiment = {'Positive': round(pos/total*100, 1), 'Negative': round(neg/total*100, 1), 'Neutral': round(neu/total*100, 1)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#25D366','#FF5252','#FFC107']
ax1.bar(sentiment.keys(), sentiment.values(), color=colors)
ax1.set_title('Sentiment Distribution (%)')
ax1.set_ylabel('Percentage')

ax2.pie(sentiment.values(), labels=sentiment.keys(), autopct='%1.1f%%',
        colors=colors, startangle=90, wedgeprops=dict(edgecolor='#111', linewidth=2))
ax2.set_title('Sentiment Share')

plt.tight_layout(); plt.show()
print(sentiment)


## 11. 📊 Advanced Insights

In [ ]:
# Message type breakdown
print("=== Message Types ===")
print(f"Text messages  : {(~df['is_media'] & ~df['is_deleted']).sum():,}")
print(f"Media messages : {df['is_media'].sum():,}")
print(f"Deleted msgs   : {df['is_deleted'].sum():,}")

# Chat streak
dates = sorted(df['only_date'].unique())
max_streak, cur_streak = 1, 1
for i in range(1, len(dates)):
    if (dates[i] - dates[i-1]).days == 1:
        cur_streak += 1
        max_streak = max(max_streak, cur_streak)
    else:
        cur_streak = 1

print(f"\n=== Chat Streaks ===")
print(f"Longest streak : {max_streak} consecutive days")
print(f"Current streak : {cur_streak} consecutive days")

# Most active date
best = df.groupby('only_date').count()['message'].idxmax()
best_count = df.groupby('only_date').count()['message'].max()
print(f"\n=== Peak Activity ===")
print(f"Most active date : {best} ({best_count:,} messages)")
total_days = (df['only_date'].max() - df['only_date'].min()).days or 1
print(f"Avg msgs/day     : {len(df)/total_days:.1f}")


In [ ]:
# Response time analysis
df2 = df[df['user'] != 'group_notification'].sort_values('date').copy()
df2['prev_date'] = df2['date'].shift(1)
df2['prev_user'] = df2['user'].shift(1)
df2['gap_min'] = (df2['date'] - df2['prev_date']).dt.total_seconds() / 60

responses = df2[(df2['user'] != df2['prev_user']) & (df2['gap_min'] < 60) & (df2['gap_min'] > 0)]
resp_avg = responses.groupby('user')['gap_min'].mean().round(1).sort_values().head(10)

if not resp_avg.empty:
    plt.figure(figsize=(10, 4))
    plt.barh(resp_avg.index[::-1], resp_avg.values[::-1], color='#FF9800')
    plt.title('Average Response Time by User (minutes, responses < 60 min)')
    plt.xlabel('Minutes')
    plt.tight_layout(); plt.show()
    print(resp_avg)


## 12. ✅ Summary

This notebook covered:

| Section | Analysis |
|---|---|
| Preprocessing | Multi-format date parsing, feature engineering |
| Timelines | Daily, monthly, quarterly, yearly trends |
| Activity | Heatmap, hourly, day-of-week, month patterns |
| Users | Ranking, message share, avg length, emoji usage |
| Text | Word cloud, top words, bigrams |
| Emoji | Top emojis, per-user usage |
| Links | Domain frequency |
| Sentiment | Rule-based positive/negative/neutral |
| Insights | Streaks, peak day, response times, message types |

---
*Export the chat from WhatsApp → ⋮ → More → Export chat → Without media*
